<a href="https://colab.research.google.com/github/MarcusLNF/Code-Wars/blob/main/especificacao_trab_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Trabalho 1: Diferenciação Automática com Grafos Computacionais

## Informações Gerais

- Data de Entrega: 28/11/2025
- Pontuação: 10 pontos
- O trabalho deve ser feito individualmente.
- A entrega do trabalho deve ser realizada via sistema testr.



## Especificação

⚠️ *Esta explicação assume que você leu e entendeu os slides sobre grafos computacionais.*

O trabalho consiste em implementar um sistema de diferenciação automática usando grafos computacionais e utilizar este sistema para resolver um conjunto de problemas.

Para isto, devem ser definidos um tipo Tensor para representar dados (similares aos arrays do numpy) e operações (e.g., soma, subtração, etc.) que geram tensores como saída.

Sempre que uma operação é realizada, é armazenado no tensor de saída referências para os seus pais, isto é, os valores usados como entrada para a operação.


### Imports

In [ ]:
from typing import Optional, Union, Any
from collections.abc import Iterable
from abc import ABC, abstractmethod
import numbers

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

### Classe NameManager

A classe NameManager provê uma forma conveniente de dar nomes intuitivos para tensores que resultam de operações. A idéia é tornar mais fácil para o usuário das demais classes qual operação gerou qual tensor. Ela provê os seguintes métodos públicos:

- reset(): reinicia o sistema de gestão de nomes.
- new(<basename>: str): retorna um nome único a partir do nome de base passado como argumento.
  
Como indicado no exemplo abaixo da classe, a idéia geral é que uma sequência de operações é feita, os nomes dos tensores sejam os nomes das operações seguidos de um número. Se forem feitas 3 operações de soma e uma de multiplicação, seus tensores de saída terão os nomes "add:0", "add:1", "add:2" e "prod:0".

In [ ]:

class NameManager:
    _counts = {}

    @staticmethod
    def reset():
        NameManager._counts = {}

    @staticmethod
    def _count(name):
        if name not in NameManager._counts:
            NameManager._counts[name] = 0
        count = NameManager._counts[name]
        return count

    @staticmethod
    def _inc_count(name):
        assert name in NameManager._counts, f'Name {name} is not registered.'
        NameManager._counts[name] += 1

    @staticmethod
    def new(name: str):
        count = NameManager._count(name)
        tensor_name = f"{name}:{count}"
        NameManager._inc_count(name)
        return tensor_name

# exemplo de uso
print(NameManager.new('add'))
print(NameManager.new('in'))
print(NameManager.new('add'))
print(NameManager.new('add'))
print(NameManager.new('in'))
print(NameManager.new('prod'))

NameManager.reset()

add:0
in:0
add:1
add:2
in:1
prod:0


### Classe Tensor

Deve ser criada uma classe `Tensor` representando um array multidimensional.

In [ ]:

class Tensor:
    """Classe representando um array multidimensional.

    Atributos:

    - _arr  (privado): dados internos do tensor como
        um array do numpy com 2 dimensões (ver Regras)

    - _parents (privado): lista de tensores que foram
        usados como argumento para a operação que gerou o
        tensor. Será vazia se o tensor foi inicializado com
        valores diretamente. Por exemplo, se o tensor foi
        resultado da operação a + b entre os tensores a e b,
        _parents = [a, b].

    - requires_grad (público): indica se devem ser
        calculados gradientes para o tensor ou não.

    - grad (público): Tensor representando o gradiente.

    """

    def __init__(self,
                 # Dados do tensor. Além dos tipos listados,
                 # arr também pode ser do tipo Tensor.
                 arr: Union[np.ndarray, list, numbers.Number, Any],
                 # Entradas da operacao que gerou o tensor.
                 # Deve ser uma lista de itens do tipo Tensor.
                 parents: list[Any] = [],
                 # se o tensor requer o calculo de gradientes ou nao
                 requires_grad: bool = True,
                 # nome do tensor
                 name: str = '',
                 # referência para um objeto do tipo Operation (ou
                 # subclasse) indicando qual operação gerou este
                 # tensor. Este objeto também possui um método
                 # para calcular a derivada da operação.
                 operation=None):
        """Construtor

        O construtor deve permitir a criacao de tensores das seguintes formas:

            # a partir de escalares
            x = Tensor(3)

            # a partir de listas
            x = Tensor([1,2,3])

            # a partir de arrays
            x = Tensor(np.array([1,2,3]))

            # a partir de outros tensores (construtor de copia)
            x = Tensor(Tensor(np.array([1,2,3])))

        Para isto, as seguintes regras devem ser obedecidas:

        - Se o argumento arr não for um array do numpy,
            ele deve ser convertido em um. Defina o dtype do
            array como float de forma a permitir que NÃO seja
            necessário passar constantes float como Tensor(3.0),
            mas possamos criar um tensor apenas com Tensor(3).

        - O atributo _arr deve ser uma matriz, isto é,
            ter 2 dimensões (ver Regras).

        - Se o argumento arr for um Tensor, ele deve ser
            copiado (cuidado com cópias por referência).

        - Se arr for um array do numpy com 1 dimensão,
            ele deve ser convertido em uma matriz coluna.

        - Se arr for um array do numpy com dimensão maior
            que 2, deve ser lançada uma exceção.

        - Tensores que não foram produzidos como resultado
            de uma operação não têm pais nem operação.
            Os nomes destes tensores devem seguir o formato in:3.
        """
        if isinstance(arr, Tensor):
            arr = arr._arr.copy()

        if not isinstance(arr, np.ndarray):
            arr = np.array(arr, dtype=float)

        if arr.ndim == 0:
            arr = arr.reshape(1, 1)
            requires_grad = False

        if arr.ndim == 1:
            arr = arr.reshape(-1, 1)

        if arr.ndim > 2:
            raise ValueError("Tensor deve ter no máximo 2 dimensões.")

        self._arr = arr
        self._parents = parents.copy()
        self.requires_grad = requires_grad
        self.operation = operation

        if name:
            self._name = name
        else:
            if parents == [] and operation is None:
                self._name = f"in:{id(self) % 10000}"
            else:
                self._name = ""

        self.grad = None


    def zero_grad(self):
        """Reinicia o gradiente com zero"""
        self.grad = Tensor(np.zeros_like(self._arr), requires_grad=False)

    def numpy(self):
        """Retorna o array interno"""
        return self._arr

    def __repr__(self):
        """Permite visualizar os dados do tensor como string"""
        return f"Tensor({self._arr}, name={self._name}, shape={self._arr.shape})"

    def backward(self, my_grad=None):
        """Método usado tanto iniciar o processo de
        diferenciação automática, quanto por um filho
        para enviar o gradiente do pai. No primeiro
        caso, o argumento my_grad não será passado.
        """
        if my_grad is None:
            my_grad = Tensor(np.ones_like(self._arr), requires_grad=False)

        if self.grad is None:
            self.grad = my_grad
        else:
            self.grad = Tensor(self.grad._arr + my_grad._arr, requires_grad=False)

        if self.operation is None:
            return

        parent_grads = self.operation.backward(self, my_grad)

        for parent, grad in zip(self._parents, parent_grads):
            parent.backward(grad)


### Interface de  Operações

A classe abaixo define a interface que as operações devem implementar. Ela não precisa ser modificada, mas pode, caso queira.

In [ ]:

class Op(ABC):

    def backward(self, child, back_grad):
        """
        child = tensor resultante da operação
        back_grad = gradiente vindo do filho
        """
        return self.grad(back_grad, *child._parents)

    def _ts(self, args):
        """Converte entradas para Tensores se necessário"""
        ts = []
        for a in args:
            if isinstance(a, Tensor):
                ts.append(a)
            else:
                ts.append(Tensor(a))
        return ts

    @abstractmethod
    def __call__(self, *args, **kwargs) -> Tensor:

        """Realiza a operação usando as entradas e
            retorna o tensor resultado. O método deve
            garantir que o atributo parents do tensor
            de saída seja uma lista de tensores."""

    @abstractmethod
    def grad(self, back_grad: Tensor, *args, **kwargs) -> list[Tensor]:
        """Retorna os gradientes dos pais em como tensores.

        Arguments:

        - back_grad: Derivada parcial em relação à saída
            da operação backpropagada pelo filho.

        - args: variaveis de entrada da operacao (pais)
            como tensores.

        - O nome dos tensores de gradiente devem ter o
            nome da operacao seguido de '_grad'.
        """


### Implementação das Operações

Operações devem herdar de `Op` e implementar os métodos `__call__` e `grad`.

Pelo menos as seguintes operações devem ser implementadas:



In [ ]:

class Add(Op):
    def __call__(self, *args, **kwargs) -> Tensor:
        args = self._ts(args)
        result = args[0].numpy() + args[1].numpy()
        return Tensor(result, parents=args, name=NameManager.new('add'), operation=self)

    def grad(self, back_grad, *args, **kwargs) -> list[Tensor]:
        g1 = Tensor(back_grad.numpy(), name=NameManager.new('add_grad'))
        g2 = Tensor(back_grad.numpy(), name=NameManager.new('add_grad'))

        return [g1, g2]

# Instancia a classe. O objeto passa a poder ser usado como uma funcao
add = Add()

In [ ]:
class Sub(Op):
    def __call__(self, *args, **kwargs) -> Tensor:
        args = self._ts(args)
        result = args[0].numpy() - args[1].numpy()
        return Tensor(result, parents=args, name=NameManager.new('sub'), operation=self)

    def grad(self, back_grad: Tensor, *args, **kwargs) -> list[Tensor]:
        grad_a = Tensor(back_grad.numpy(), name=NameManager.new('sub_grad'))
        grad_b = Tensor(-back_grad.numpy(), name=NameManager.new('sub_grad'))
        return [grad_a, grad_b]


# Instancia a classe
sub = Sub()


In [ ]:

class Prod(Op):
    def __call__(self, *args, **kwargs) -> Tensor:
        args = self._ts(args)
        result = args[0].numpy() * args[1].numpy()
        return Tensor(result, parents=args, name=NameManager.new('prod'), operation=self)

    def grad(self, back_grad: Tensor, *args, **kwargs) -> list[Tensor]:
        a, b = args
        grad_a = Tensor(b.numpy() * back_grad.numpy(), name=NameManager.new('prod_grad'))
        grad_b = Tensor(a.numpy() * back_grad.numpy(), name=NameManager.new('prod_grad'))
        return [grad_a, grad_b]

# Instancia a classe. O objeto passa a poder ser usado como uma funcao
prod = Prod()

In [ ]:

class Sin(Op):
    def __call__(self, *args, **kwargs) -> Tensor:
        args = self._ts(args)
        result = np.sin(args[0].numpy())
        return Tensor(result, parents=args, name=NameManager.new('sin'), operation=self)

    def grad(self, back_grad: Tensor, *args, **kwargs) -> list[Tensor]:
        t = args[0]
        g = np.cos(t.numpy()) * back_grad.numpy()
        return [Tensor(g, name=NameManager.new('sin_grad'))]


# Instancia a classe. O objeto passa a poder ser usado como uma funcao
sin = Sin()

In [ ]:

class Cos(Op):
    def __call__(self, *args, **kwargs) -> Tensor:
        args = self._ts(args)
        result = np.cos(args[0].numpy())
        return Tensor(result, parents=args, name=NameManager.new('cos'), operation=self)

    def grad(self, back_grad: Tensor, *args, **kwargs) -> list[Tensor]:
        t = args[0]
        g = -np.sin(t.numpy()) * back_grad.numpy()
        return [Tensor(g, name=NameManager.new('cos_grad'))]


# Instancia a classe. O objeto passa a poder ser usado como uma funcao
cos = Cos()

In [ ]:

class Sum(Op):
    def __call__(self, *args, **kwargs) -> Tensor:
        args = self._ts(args)
        result = np.sum(args[0].numpy())
        return Tensor(result, parents=args, name=NameManager.new('my_sum'), operation=self)

    def grad(self, back_grad: Tensor, *args, **kwargs) -> list[Tensor]:
        t = args[0]
        g = np.ones_like(t.numpy()) * back_grad.numpy()
        return [Tensor(g,name=NameManager.new('my_sum_grad'))]

# Instancia a classe. O objeto passa a poder ser usado como uma funcao
# ⚠️ vamos chamar de my_sum porque python ja possui uma funcao sum
my_sum = Sum()

In [ ]:

class Mean(Op):
    def __call__(self, *args, **kwargs) -> Tensor:
        args = self._ts(args)
        result = np.mean(args[0].numpy())
        return Tensor(result, parents=args, name=NameManager.new('mean'), operation=self)

    def grad(self, back_grad: Tensor, *args, **kwargs) -> list[Tensor]:
        t = args[0]
        N = t.numpy().size
        g = np.ones_like(t.numpy()) * (back_grad.numpy() / N)
        return [Tensor(g, name=NameManager.new('mean_grad'))]

# Instancia a classe. O objeto passa a poder ser usado como uma funcao
mean = Mean()

In [ ]:

class Square(Op):
    def __call__(self, *args, **kwargs) -> Tensor:
        args = self._ts(args)
        result = np.square(args[0].numpy())
        return Tensor(result, parents=args, name=NameManager.new('square'), operation=self)

    def grad(self, back_grad: Tensor, *args, **kwargs) -> list[Tensor]:
        t = args[0]
        g = 2 * t.numpy() * back_grad.numpy()
        return [Tensor(g, name=NameManager.new('square_grad'))]

# Instancia a classe. O objeto passa a poder ser usado como uma funcao
square = Square()

In [ ]:

class MatMul(Op):
    """MatMul(A, B): multiplicação de matrizes

    C = A @ B
    de/dA = de/dc @ B^T
    de/dB = A^T @ de/dc

    """

class MatMul(Op):
    def __call__(self, *args, **kwargs) -> Tensor:
        args = self._ts(args)
        a = args[0].numpy()
        b = args[1].numpy()

        assert len(a.shape) == 2 and len(b.shape) == 2
        assert a.shape[1] == b.shape[0]

        result = np.matmul(a, b)
        return Tensor(result, parents=args, name=NameManager.new('matmul'), operation=self)

    def grad(self, back_grad: Tensor, *args, **kwargs) -> list[Tensor]:
        a = args[0].numpy()
        b = args[1].numpy()
        grad_a = back_grad.numpy() @ b.T
        grad_b = a.T @ back_grad.numpy()

        assert grad_a.shape == a.shape
        assert grad_b.shape == b.shape

        return [Tensor(grad_a, name=NameManager.new('matmul_grad')),
                Tensor(grad_b, name=NameManager.new('matmul_grad'))]

# Instancia a classe. O objeto passa a poder ser usado como uma funcao
matmul = MatMul()

In [ ]:

class Exp(Op):
    def __call__(self, *args, **kwargs) -> Tensor:
        args = self._ts(args)
        result = np.exp(args[0].numpy())
        return Tensor(result, parents=args, name=NameManager.new('exp'), operation=self)

    def grad(self, back_grad: Tensor, *args, **kwargs) -> list[Tensor]:
        t = args[0]
        g = np.exp(t.numpy()) * back_grad.numpy()

        return [Tensor(g, name=NameManager.new('exp_grad'))]


# Instancia a classe. O objeto passa a poder ser usado como uma funcao
exp = Exp()

In [ ]:

class ReLU(Op):
    def __call__(self, *args, **kwargs) -> Tensor:
        args = self._ts(args)
        result = np.maximum(0, args[0].numpy())
        return Tensor(result, parents=args, name=NameManager.new('relu'), operation=self)

    def grad(self, back_grad: Tensor, *args, **kwargs) -> list[Tensor]:
        t = args[0]
        g = (t.numpy() > 0) * back_grad.numpy()
        return [Tensor(g, name=NameManager.new('relu_grad'))]

# Instancia a classe. O objeto passa a poder ser usado como uma funcao
relu = ReLU()

In [ ]:

class Sigmoid(Op):
    def __call__(self, *args, **kwargs) -> Tensor:
        args = self._ts(args)
        result = 1 / (1 + np.exp(-args[0].numpy()))
        return Tensor(result, parents=args, name=NameManager.new('sigmoid'), operation=self)

    def grad(self, back_grad: Tensor, *args, **kwargs) -> list[Tensor]:
        t = args[0]
        sig = 1 / (1 + np.exp(-t.numpy()))
        g = sig * (1 - sig) * back_grad.numpy()
        return [Tensor(g, name=NameManager.new('sigmoid_grad'))]


# Instancia a classe. O objeto passa a poder ser usado como uma funcao
sigmoid = Sigmoid()

In [ ]:

class Tanh(Op):
    def __call__(self, *args, **kwargs) -> Tensor:
        args = self._ts(args)
        result = np.tanh(args[0].numpy())
        return Tensor(result, parents=args, name=NameManager.new('tanh'), operation=self)

    def grad(self, back_grad: Tensor, *args, **kwargs) -> list[Tensor]:
        t = args[0]
        tanh_val = np.tanh(t.numpy())
        g = (1 - tanh_val**2) * back_grad.numpy()
        return [Tensor(g, name=NameManager.new('tanh_grad'))]


# Instancia a classe. O objeto passa a poder ser usado como uma funcao
tanh = Tanh()

In [ ]:

class Softmax(Op):
    def __call__(self, *args, **kwargs) -> Tensor:
        args = self._ts(args)

        v = args[0].numpy().reshape(-1, 1)
        shifted = v - np.max(v)
        expv = np.exp(shifted)
        result = expv / np.sum(expv)

        return Tensor(result, parents=args, name=NameManager.new('softmax'), operation=self)

    def grad(self, back_grad: Tensor, *args, **kwargs) -> list[Tensor]:
        t = args[0]
        v = t.numpy().reshape(-1, 1)
        shifted = v - np.max(v)
        expv = np.exp(shifted)
        soft = expv / np.sum(expv)

        ds = np.diagflat(soft.flatten()) - np.dot(soft, soft.T)

        back = back_grad.numpy().reshape(-1, 1)
        g = np.dot(ds, back)
        g = g.reshape(soft.shape)


        return [Tensor(g, name=NameManager.new('softmax_grad'))]


# Instancia a classe. O objeto passa a poder ser usado como uma funcao
softmax = Softmax()


### ‼️ Regras e Pontos de Atenção‼️

- Vamos fazer a hipótese simplificadora que Tensores devem ser sempre matrizes. Por exemplo, o escalar 2 deve ser armazado em `_arr` como a matriz `[[2]]`. De forma similar, a lista `[1, 2, 3]` deve ser armazenada em `_arr` como em uma matriz coluna.

- Devem ser realizados `asserts` nas operações para garantir que os shapes dos operandos fazem sentido. Esta verificação também deve ser feita depois das operações que manipulam gradientes de tensores.

- Devem ser respeitados os nomes dos atributos, métodos e classes para viabilizar os testes automáticos.

- Gradientes devem ser calculados usando uma passada pelo grafo computacional.

- Os gradientes devem ser somados e não substituídos nas chamadas de  backward. Isto vai permitir que os gradientes sejam acumulados entre amostras do dataset e que os resultados sejam corretos mesmo em caso de ramificações e junções no grafo computacional.

- Lembre-se de zerar os gradientes após cada passo de gradient descent (atualização dos parâmetros).


## Testes Básicos

Estes testes avaliam se a derivada da função está sendo calculada corretamente, mas em muitos casos **não** avaliam se os gradientes backpropagados estão sendo incorporados corretamente. Esta avaliação será feita nos problemas da próxima seção.

Operador de Soma

In [ ]:
# add

a = Tensor([1.0, 2.0, 3.0])
b = Tensor([4.0, 5.0, 6.0])
c = add(a, b)
d = add(c, 3.0)
d.backward()

# esperado: matrizes coluna contendo 1
print(a.grad)
print(b.grad)


Tensor([[1.]
 [1.]
 [1.]], name=add_grad:2, shape=(3, 1))
Tensor([[1.]
 [1.]
 [1.]], name=add_grad:3, shape=(3, 1))


Operador de Subtração

In [ ]:
# sub

a = Tensor([1.0, 2.0, 3.0])
b = Tensor([4.0, 5.0, 6.0])
c = sub(a, b)
d = sub(c, 3.0)
d.backward()

# esperado: matrizes coluna contendo 1 e -1
print(a.grad)
print(b.grad)


Tensor([[1.]
 [1.]
 [1.]], name=sub_grad:2, shape=(3, 1))
Tensor([[-1.]
 [-1.]
 [-1.]], name=sub_grad:3, shape=(3, 1))


Operador de Produto

In [ ]:
# prod

a = Tensor([1.0, 2.0, 3.0])
b = Tensor([4.0, 5.0, 6.0])
c = prod(a, b)
d = prod(c, 3.0)
d.backward()

# esperado: [12, 15, 18]^T
print(a.grad)
# esperado: [3, 6, 9]^T
print(b.grad)


Tensor([[12.]
 [15.]
 [18.]], name=prod_grad:2, shape=(3, 1))
Tensor([[3.]
 [6.]
 [9.]], name=prod_grad:3, shape=(3, 1))


Operadores trigonométricos

In [ ]:
# sin e cos

a = Tensor([np.pi, 0, np.pi/2])
b = sin(a)
c = cos(a)
d = my_sum(add(b, c))
d.backward()

# esperado: [-1, 1, -1]^T
print(a.grad)

Tensor([[-1.]
 [ 1.]
 [-1.]], name=in:2560, shape=(3, 1))


In [ ]:
# Sum

a = Tensor([3.0, 1.0, 0.0, 2.0])
b = add(prod(a, 3.0), a)
c = my_sum(b)
c.backward()

# esperado: [4, 4, 4, 4]^T
print(a.grad)


Tensor([[4.]
 [4.]
 [4.]
 [4.]], name=in:864, shape=(4, 1))


In [ ]:
# Mean

a = Tensor([3.0, 1.0, 0.0, 2.0])
b = mean(a)
b.backward()

# esperado: [0.25, 0.25, 0.25, 0.25]^T
print(a.grad)


Tensor([[0.25]
 [0.25]
 [0.25]
 [0.25]], name=mean_grad:0, shape=(4, 1))


In [ ]:
# Square

a = Tensor([3.0, 1.0, 0.0, 2.0])
b = square(a)

# esperado: [9, 1, 0, 4]^T
print(b)

b.backward()

# esperado: [6, 2, 0, 4]
print(a.grad)

Tensor([[9.]
 [1.]
 [0.]
 [4.]], name=square:0, shape=(4, 1))
Tensor([[6.]
 [2.]
 [0.]
 [4.]], name=square_grad:0, shape=(4, 1))


In [ ]:
# matmul

W = Tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
    [7.0, 8.0, 9.0]
])

v = Tensor([1.0, 2.0, 3.0])

z = matmul(W, v)

# esperado: [14, 32, 50]^T
print(z)

z.backward()

# esperado:
# [1, 2, 3]
# [1, 2, 3]
# [1, 2, 3]
print(W.grad)

# esperado: [12, 15, 18]^T
print(v.grad)


Tensor([[14.]
 [32.]
 [50.]], name=matmul:0, shape=(3, 1))
Tensor([[1. 2. 3.]
 [1. 2. 3.]
 [1. 2. 3.]], name=matmul_grad:0, shape=(3, 3))
Tensor([[12.]
 [15.]
 [18.]], name=matmul_grad:1, shape=(3, 1))


In [ ]:
# Exp

v = Tensor([1.0, 2.0, 3.0])
w = exp(v)

# esperado: [2.718..., 7.389..., 20.085...]^T
print(w)

w.backward()

# esperado: [2.718..., 7.389..., 20.085...]^T
print(v.grad)

Tensor([[ 2.71828183]
 [ 7.3890561 ]
 [20.08553692]], name=exp:0, shape=(3, 1))
Tensor([[ 2.71828183]
 [ 7.3890561 ]
 [20.08553692]], name=exp_grad:0, shape=(3, 1))


In [ ]:
# Relu

v = Tensor([-1.0, 0.0, 1.0, 3.0])
w = relu(v)

# esperado: [0, 0, 1, 3]^T
print(w)

w.backward()

# esperado: [0, 0, 1, 1]^T
print(v.grad)

Tensor([[0.]
 [0.]
 [1.]
 [3.]], name=relu:0, shape=(4, 1))
Tensor([[0.]
 [0.]
 [1.]
 [1.]], name=relu_grad:0, shape=(4, 1))


In [ ]:
# Sigmoid

v = Tensor([-1.0, 0.0, 1.0, 3.0])
w = sigmoid(v)

# esperado: [0.268.., 0.5, 0.731.., 0.952..]^T
print(w)

w.backward()

# esperado: [0.196..., 0.25, 0.196..., 0.045...]^T
print(v.grad)

Tensor([[0.26894142]
 [0.5       ]
 [0.73105858]
 [0.95257413]], name=sigmoid:0, shape=(4, 1))
Tensor([[0.19661193]
 [0.25      ]
 [0.19661193]
 [0.04517666]], name=sigmoid_grad:0, shape=(4, 1))


In [ ]:
# Tanh

v = Tensor([-1.0, 0.0, 1.0, 3.0])
w = tanh(v)

# esperado: [[-0.76159416, 0., 0.76159416, 0.99505475]^T
print(w)

w.backward()

# esperado: [0.41997434, 1., 0.41997434, 0.00986604]^T
print(v.grad)

Tensor([[-0.76159416]
 [ 0.        ]
 [ 0.76159416]
 [ 0.99505475]], name=tanh:0, shape=(4, 1))
Tensor([[0.41997434]
 [1.        ]
 [0.41997434]
 [0.00986604]], name=tanh_grad:0, shape=(4, 1))


In [ ]:
# Softmax

x = Tensor([-3.1, 0.5, 1.0, 2.0])
y = softmax(x)

# esperado: [0.00381737, 0.13970902, 0.23034123, 0.62613238]^T
print(y)

# como exemplo, calcula o MSE para um target vector
diff = sub(y, [1, 0, 0, 0])
sq = square(diff)
a = mean(sq)

# esperado: 0.36424932
print("MSE:", a)

a.backward()

# esperado: [-0.00278095, -0.02243068, -0.02654377, 0.05175539]^T
print(x.grad)



Tensor([[0.00381737]
 [0.13970902]
 [0.23034123]
 [0.62613238]], name=softmax:0, shape=(4, 1))
MSE: Tensor([[0.36424932]], name=mean:1, shape=(1, 1))
Tensor([[-0.00278095]
 [-0.02243068]
 [-0.02654377]
 [ 0.05175539]], name=softmax_grad:0, shape=(4, 1))



## Referências

### Principais

- [Build your own pytorch](https://www.peterholderrieth.com/blog/2023/Build-Your-Own-Pytorch-1-Computation-Graphs/)
- [Build your own Pytorch - 2: Backpropagation](https://www.peterholderrieth.com/blog/2023/Build-Your-Own-Pytorch-2-Autograd/)
- [Build your own PyTorch - 3: Training a Neural Network with self-made AD software](https://www.peterholderrieth.com/blog/2023/Build-Your-Own-Pytorch-3-Build-Classifier/)
- [Pytorch: A Gentle Introduction to torch.autograd](https://docs.pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html)
- [Automatic Differentiation with torch.autograd](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html)

### Secundárias

- [Tom Roth: Building a computational graph: part 1](https://tomroth.dev/compgraph1/)
- [Tom Roth: Building a computational graph: part 2](https://tomroth.dev/compgraph2/)
- [Tom Roth: Building a computational graph: part 3](https://tomroth.dev/compgraph3/)
- [Roger Grosse (Toronto) class on Automatic Differentiation](https://www.cs.toronto.edu/~rgrosse/courses/csc321_2018/slides/lec10.pdf)
- [Computational graphs and gradient flows](https://simple-english-machine-learning.readthedocs.io/en/latest/neural-networks/computational-graphs.html)
- [Colah Visual Blog: Backprop](https://colah.github.io/posts/2015-08-Backprop/)
- [Towards Data Science: Automatic Differentiation (AutoDiff): A Brief Intro with Examples](https://towardsdatascience.com/automatic-differentiation-autodiff-a-brief-intro-with-examples-3f3d257ffe3b/)
- [A Hands-on Introduction to Automatic Differentiation - Part 1](https://mostafa-samir.github.io/auto-diff-pt1/)
- [Build Your own Deep Learning Framework - A Hands-on Introduction to Automatic Differentiation - Part 2](https://mostafa-samir.github.io/auto-diff-pt1/)
